# Lab medicine: data-driven reference intervals and within-person variation

**For:** pathology and laboratory medicine, biorepository.

**Questions:**
1. From a real-world cohort, what reference intervals would we derive for common analytes, and how
   do they compare with the intervals the performing lab reported?
2. Over 5 annual visits, how much does each analyte vary *within a person*, and how often does a
   "healthy" person cross a reference limit?

**Requirements:** a workspace with the Project Baseline Health Study (Limited Tier) data collection added (resources `screener`, `crf`, `analysis`, `externallab`, `corelabreads`, `appsurveys`).

**Data:** Project Project Baseline (Limited Tier) external labs (88 tests, about 670k results, 5 visits).
Adults only. Pediatric intervals would need pediatric data; the method carries over unchanged.

In [ ]:
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

bq = bigquery.Client()

_resolved = {}

def tbl(dataset, table):
    """Look up a workspace dataset by resource name: tbl('crf', 'MH') -> `<project>.crf.MH`."""
    if dataset not in _resolved:
        _resolved[dataset] = subprocess.run(["wb", "resolve", "--name", dataset],
                                            capture_output=True, text=True, check=True).stdout.strip()
    return f"`{_resolved[dataset]}.{table}`"

def sql(q):
    return bq.query(q).to_dataframe()

ANALYTES = ["HEMOGLOBIN", "PLATELET COUNT", "ALKALINE PHOSPHATASE", "ALAT (SGPT)",
            "CREATININE ENZ SERUM", "POTASSIUM", "HEMOGLOBIN A1C", "CRP HIGH SENSITIVITY"]

## 1. Pull results plus an a priori "reference population" filter
Exclusions (CLSI EP28-style, a priori): known diabetes, hypertension, or CKD at enrollment.

In [ ]:
labs = sql(f"""
SELECT l.USUBJID, l.VISIT, l.VISITNUM, l.rtcname AS analyte,
       SAFE_CAST(l.siurslt AS FLOAT64) AS value, l.siuunit AS unit,
       SAFE_CAST(l.siulow AS FLOAT64) AS lab_low, SAFE_CAST(l.siuhigh AS FLOAT64) AS lab_high,
       d.SEX AS sex, SAFE_CAST(d.age_at_enrollment AS INT64) AS age,
       (x.der_hx_diab = 0 AND x.der_hx_htn = 0 AND x.der_hx_ckd = 0) AS reference_pop
FROM {tbl('externallab', 'CLABS')} l
JOIN {tbl('screener', 'DM')} d USING (USUBJID)
JOIN {tbl('analysis', 'DIAGNOSES')} x ON x.USUBJID = l.USUBJID AND x.VISIT = 'STUDY_START'
WHERE l.rtcname IN ({", ".join(repr(a) for a in ANALYTES)})
""")
labs = labs.dropna(subset=["value"])
print(f"{len(labs):,} results · {labs.USUBJID.nunique():,} participants · "
      f"reference population: {labs[labs.reference_pop].USUBJID.nunique():,}")

## 2. Nonparametric reference intervals (2.5th–97.5th percentile, bootstrap 90% CI), by sex
Enrollment visit only, one result per person.

In [ ]:
rng = np.random.default_rng(0)

def ri(values, n_boot=500):
    v = values.to_numpy()
    boots = np.array([np.percentile(rng.choice(v, len(v)), [2.5, 97.5]) for _ in range(n_boot)])
    lo, hi = np.percentile(v, [2.5, 97.5])
    (lo_l, hi_l), (lo_h, hi_h) = np.percentile(boots, [5, 95], axis=0).T
    return lo, hi, f"{lo_l:.3g}–{lo_h:.3g}", f"{hi_l:.3g}–{hi_h:.3g}"

base = labs[(labs.VISIT == "STUDY_START") & labs.reference_pop].drop_duplicates(["USUBJID", "analyte"])
rows = []
for (a, sex), d in base.groupby(["analyte", "sex"]):
    lo, hi, lo_ci, hi_ci = ri(d.value)
    rows.append({"analyte": a, "sex": sex, "n": len(d), "unit": d.unit.mode()[0],
                 "derived_low": round(lo, 3), "derived_high": round(hi, 3),
                 "low 90% CI": lo_ci, "high 90% CI": hi_ci,
                 "lab_low": d.lab_low.mode().iloc[0] if d.lab_low.notna().any() else np.nan,
                 "lab_high": d.lab_high.mode().iloc[0] if d.lab_high.notna().any() else np.nan})
ri_table = pd.DataFrame(rows)
ri_table

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, a in zip(axes.flat, ANALYTES):
    for sex, color in [("Female", "#c0504d"), ("Male", "#3b6ea5")]:
        d = base[(base.analyte == a) & (base.sex == sex)].value
        ax.hist(d.clip(upper=d.quantile(.995)), bins=40, alpha=.45, color=color, label=sex)
        r = ri_table[(ri_table.analyte == a) & (ri_table.sex == sex)].iloc[0]
        for x in (r.derived_low, r.derived_high):
            ax.axvline(x, color=color, lw=1.5)
        for x in (r.lab_low, r.lab_high):
            if pd.notna(x) and x > 0:
                ax.axvline(x, color=color, lw=1, ls=":")
    ax.set_title(f"{a} ({base[base.analyte == a].unit.mode()[0]})", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
axes.flat[0].legend(fontsize=8)
fig.suptitle("Reference population at enrollment: solid = derived 2.5/97.5th percentile, dotted = lab-reported interval")
plt.tight_layout(); plt.show()

## 3. Within-person variation across 5 annual visits
For people in the reference population with ≥3 results: the median within-person CV (biological plus
analytical) and the resulting reference change value (RCV, 95%, two-sided) ≈ √2 × 1.96 × CV.
Also: how often does a "healthy" person fall outside the lab interval at least once?

In [ ]:
long = labs[labs.reference_pop]
per_person = (long.groupby(["analyte", "USUBJID"])
              .agg(n=("value", "size"), mean=("value", "mean"), sd=("value", "std"))
              .reset_index())
out_flags = (long.assign(out=lambda d: (d.value < d.lab_low) | (d.value > d.lab_high))
             .groupby(["analyte", "USUBJID"]).out.any().reset_index())
per_person = per_person.merge(out_flags, on=["analyte", "USUBJID"])
per_person = per_person[(per_person.n >= 3) & (per_person["mean"] > 0)]
per_person["cv"] = per_person.sd / per_person["mean"] * 100

variation = (per_person.groupby("analyte")
             .agg(people=("USUBJID", "nunique"), median_within_person_CV_pct=("cv", "median"),
                  pct_ever_outside_lab_interval=("out", lambda s: s.mean() * 100))
             .round(1))
variation["approx_RCV_pct"] = (np.sqrt(2) * 1.96 * variation.median_within_person_CV_pct).round(0)
variation.sort_values("median_within_person_CV_pct")

In [ ]:
example = "ALAT (SGPT)"
ids = per_person[per_person.analyte == example].sample(40, random_state=1).USUBJID
d = long[(long.analyte == example) & long.USUBJID.isin(ids)].sort_values("VISITNUM")
fig, ax = plt.subplots(figsize=(8, 4))
for _, g in d.groupby("USUBJID"):
    ax.plot(g.VISITNUM, g.value, color="#3b6ea5", alpha=.35, marker="o", ms=3)
hi = d.lab_high.mode()[0]
ax.axhline(hi, color="#c0504d", ls=":", label=f"lab upper limit ({hi:g})")
ax.set_xlabel("visit number"); ax.set_ylabel(f"{example} ({d.unit.mode()[0]})")
ax.set_title(f"{example}: 40 reference-population participants across annual visits")
ax.legend(); ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

**Try next:** partition by age band (`pd.cut(base.age, [18, 30, 45, 60, 100])`), apply a Tukey/Horn
outlier step before computing percentiles, or test whether a sex partition is warranted
(Harris–Boyd). The same code would run on pediatric results.

## Provenance

Generate information about this notebook environment and the packages installed.

In [ ]:
!date

Conda and pip installed packages:

In [ ]:
!conda env export

JupyterLab extensions:

In [ ]:
!jupyter labextension list

Number of cores:

In [ ]:
!grep ^processor /proc/cpuinfo | wc -l

Memory:

In [ ]:
!grep "^MemTotal:" /proc/meminfo

---
Copyright 2026 Verily Life Sciences LLC

Use of this source code is governed by a BSD-style
license that can be found in the LICENSE file or at
https://developers.google.com/open-source/licenses/bsd